# Tune the reward on `task_easy` with real Gemma 4 E4B IT

End-to-end run of one AAPL Aug-Sep 2023 episode through the full council
(7 frozen specialists + moderator) using the actual `google/gemma-4-E4B-it`
weights — captures every per-step reward component, then re-grades the
captured trajectory under different reward weights without re-running the
LLM. Tuning loop is sub-second per knob combination.

Outputs land in `training/runs/easy_gemma4_<timestamp>/`:
- `trace.jsonl` — full per-step actions + reward breakdown
- `sweep_results.csv` — every (weight combo, total reward, breakdown stats)
- `reward_curve.png`, `pnl_breakdown.png`, `actions.png`
- `chosen_settings.json` — the knobs you select after looking at the sweep

The notebook ALSO opportunistically runs one `corpus_*` task at the end
(if `data/corpus/` was built locally) to exercise the on-demand chart
rendering + EDGAR/yfinance headline fallback paths in the loader.

In [ ]:
import os, sys

def _is_stocker_root(p: str) -> bool:
    return os.path.isfile(os.path.join(p, 'app', 'council', 'specialists.py'))

for c in (os.getcwd(), '/content/stocker', '/workspace/stocker',
          os.path.expanduser('~/repos/stocker')):
    if _is_stocker_root(c):
        os.chdir(c); sys.path.insert(0, c); WORKDIR = c; break
else:
    raise RuntimeError('No stocker repo on disk in any expected location.')
print('Working dir:', WORKDIR)

In [ ]:
# Versions + ideal-profit sidecar + corpus availability.
import importlib.metadata as _m
from pathlib import Path

print('versions:')
for pkg in ('transformers', 'accelerate', 'peft', 'bitsandbytes', 'torchao', 'torch'):
    try:
        print(f'  {pkg:<14s} {_m.version(pkg)}')
    except _m.PackageNotFoundError:
        print(f'  {pkg:<14s} not installed')

if not Path('data/ideal_profits/task_easy.json').exists():
    print('\n[setup] ideal-profit sidecar missing for task_easy; building ...')
    !python scripts/build_ideal_profit.py
else:
    print('\n[setup] data/ideal_profits/task_easy.json present')

from app.data import corpus
if corpus.available():
    from app.core.corpus_tasks import list_corpus_task_ids
    ids = list_corpus_task_ids()
    print(f'[setup] corpus available: {len(ids)} tasks (sample: {ids[:3]} ...)')
else:
    print('[setup] corpus NOT available — final stage will be skipped.')
    print('        Build with: python scripts/build_corpus.py --quick')

In [ ]:
# Gemma 4 is Apache-2.0 (not gated) — token just lifts download rate limits.
# Mandatory when running through a tunneled kernel — Colab's secret vault
# isn't reachable from VSCode-over-cloudflared.
import os
if not os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    import getpass
    login(token=getpass.getpass('HF token (read scope is enough): '))
else:
    print('HF_TOKEN already set in env — skipping interactive login.')

In [ ]:
# Talk to a hosted HF Inference Endpoint instead of loading Gemma in-process.
# Decouples the LLM from the notebook kernel — Colab/local can flap freely
# without losing model state. Cost: ~$0.80/h on an L4 while requests in flight,
# free while scaled to zero (idle > scale-to-zero timeout).
#
# Required env vars:
#   HF_TOKEN          - read scope, already set
#   HF_ENDPOINT_URL   - e.g. https://abc123.us-east-1.aws.endpoints.huggingface.cloud
#                       (no trailing /v1, this cell appends it)
import os, time
from app.council.llm import OpenAILLMClient

endpoint = os.environ.get('HF_ENDPOINT_URL')
if not endpoint:
    raise RuntimeError(
        'HF_ENDPOINT_URL is unset. Create an Inference Endpoint for '
        'google/gemma-4-E4B-it (see notebook header) and export the URL.'
    )

gemma = OpenAILLMClient(
    base_url=endpoint.rstrip('/') + '/v1',
    api_key=os.environ['HF_TOKEN'],
    model='google/gemma-4-E4B-it',
)

# Smoke test - wakes the endpoint if it's scaled to zero (cold start ~30-90s
# the first time, <1s after).
t0 = time.time()
reply = gemma.complete(
    [{'role': 'user', 'content': 'Reply with the single word: ready.'}],
    max_tokens=8, temperature=0.0,
)
print(f'endpoint OK in {time.time()-t0:.1f}s')
print(f'  url:   {endpoint}')
print(f'  reply: {reply!r}')


In [ ]:
# Pre-cache 7 specialists' votes for every step of task_easy. Frozen
# specialists -> deterministic; on-disk cache -> resumable.
import sys, time
from datetime import datetime
from tqdm import tqdm

from app.council.runner import Council
from app.core.environment import StockerEnv

TASK = 'task_easy'

def _log(msg): print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)

council = Council(client=gemma, use_cache=True)

plan = []
env = StockerEnv(TASK)
obs = env.reset().observation
while True:
    for sp in council.specialists:
        plan.append((sp, obs, council._vote_cache_path(sp, obs).exists()))
    r = env.step({'side': 'hold', 'quantity': 0})
    if r.done: break
    obs = r.observation

todo = [(sp, obs) for sp, obs, hit in plan if not hit]
hits = sum(1 for _, _, hit in plan if hit)
_log(f'{TASK}: {len(plan)} (step,specialist) tuples — {hits} cached, {len(todo)} to compute.')

if not todo:
    _log('All cached — nothing to do.')
else:
    HEARTBEAT = 25
    t0 = time.time()
    pbar = tqdm(todo, desc='precaching', unit='vote', file=sys.stdout, dynamic_ncols=True)
    for i, (sp, obs) in enumerate(pbar, 1):
        council._cached_vote(sp, obs)
        pbar.set_postfix_str(f'{sp.name}@{obs.date}')
        if i % HEARTBEAT == 0 or i == len(todo):
            elapsed = time.time() - t0
            rate = i / max(elapsed, 1e-9)
            eta = (len(todo) - i) / max(rate, 1e-9)
            _log(f'  {i}/{len(todo)} done  rate={rate:.2f} v/s  ETA={eta/60:.1f} min')
    _log(f'Done in {(time.time()-t0)/60:.2f} min.')

In [ ]:
# Drive task_easy to completion + capture full per-step trace.
import json, time, sys
from datetime import datetime
from pathlib import Path
from tqdm import tqdm

from app.core.environment import StockerEnv
from app.core.tasks import get_task_definition

RUN_DIR = Path('training/runs') / f'easy_gemma4_{datetime.utcnow().strftime("%Y%m%d_%H%M%S")}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Run dir:', RUN_DIR)

task_def = get_task_definition(TASK)
ideal_series = list(task_def['ideal_pnl_pct_series'])
ideal_total  = float(task_def['ideal_pnl_pct_total'])

env = StockerEnv(TASK)
obs = env.reset().observation
starting_cash = env._task['starting_cash']
total_steps = env.state().total_steps

steps = []
step_index = 0
t0 = time.time()

pbar = tqdm(total=total_steps, desc='episode', unit='step', file=sys.stdout, dynamic_ncols=True)
while True:
    decision = council.run(obs)
    result = env.step(decision.action)
    info = result.info
    steps.append({
        'step_index': step_index,
        'date': obs.date,
        'price': obs.price,
        'action': {'side': decision.action.side, 'quantity': decision.action.quantity},
        'cash_after': info.get('cash'),
        'position_after': info.get('position'),
        'portfolio_after': info.get('portfolio_value'),
        'env_reward': result.reward,
        'reward_breakdown': info.get('reward_breakdown', {}),
        'invalid': 'invalid' in (info.get('trade_feedback') or ''),
        'votes': [v.model_dump() for v in decision.votes],
        'moderator_rationale': decision.rationale[:600],
    })
    step_index += 1
    pbar.update(1)
    pbar.set_postfix_str(
        f'{decision.action.side}({decision.action.quantity:+.0f}) {obs.date} '
        f'port={info.get("portfolio_value", 0):.0f}'
    )
    if result.done: break
    obs = result.observation
pbar.close()

trace = {
    'task_id': TASK,
    'starting_cash': starting_cash,
    'total_steps': total_steps,
    'ideal_pnl_pct_series': ideal_series,
    'ideal_pnl_pct_total': ideal_total,
    'buy_and_hold_value': env._buy_and_hold_value(),
    'final_portfolio': steps[-1]['portfolio_after'],
    'steps': steps,
}

(RUN_DIR / 'trace.jsonl').write_text('\n'.join(json.dumps(s) for s in steps))
(RUN_DIR / 'trace_meta.json').write_text(json.dumps({k: v for k, v in trace.items() if k != 'steps'}, indent=2))

elapsed = time.time() - t0
actions = [s['action']['side'] for s in steps]
from collections import Counter
print(f'\nepisode complete in {elapsed/60:.1f} min ({len(steps)} steps)')
print(f'  total env reward:  {sum(s["env_reward"] for s in steps):+.4f}')
print(f'  final portfolio:   {trace["final_portfolio"]:.2f}')
print(f'  buy-and-hold:      {trace["buy_and_hold_value"]:.2f}')
print(f'  action mix:        {Counter(actions)}')

In [ ]:
# Pure-Python re-grader. Replays compute_step_reward on the captured trace
# under any Settings instance. No LLM calls — sub-millisecond per step.
from app.config import Settings
from app.core.graders import compute_step_reward, compute_trajectory_bonus
from app.models import TradeAction

def replay(trace: dict, settings: Settings) -> dict:
    rewards = []
    breakdown_records = []
    for s in trace['steps']:
        action = TradeAction(**s['action'])
        result = compute_step_reward(
            action=action,
            new_portfolio=s['portfolio_after'],
            starting_cash=trace['starting_cash'],
            invalid=s['invalid'],
            step_index=s['step_index'],
            total_steps=trace['total_steps'],
            ideal_pnl_pct_series=trace['ideal_pnl_pct_series'],
            ideal_pnl_pct_total=trace['ideal_pnl_pct_total'],
            settings=settings,
        )
        breakdown_records.append(result.breakdown)
        rewards.append(result.score)
    # Final-step trajectory bonus (matches env logic).
    bonus = compute_trajectory_bonus(
        final_portfolio=trace['final_portfolio'],
        buy_and_hold_value=trace['buy_and_hold_value'],
        starting_cash=trace['starting_cash'],
    )
    rewards[-1] += bonus
    rewards = [max(-1.0, min(1.0, r)) for r in rewards]
    return {
        'rewards_per_step': rewards,
        'total_reward': sum(rewards),
        'breakdowns': breakdown_records,
        'trajectory_bonus': bonus,
    }

# Sanity: replaying with default Settings should match env exactly.
rep = replay(trace, Settings())
env_rewards = [s['env_reward'] for s in trace['steps']]
diffs = [abs(a - b) for a, b in zip(rep['rewards_per_step'], env_rewards)]
max_diff = max(diffs)
print(f'replay vs env: max abs diff = {max_diff:.2e}')
assert max_diff < 1e-5, 'replay does not match env — graders.py may have changed'
print(f'replay total reward (default settings): {rep["total_reward"]:+.4f}')
print(f'env total reward:                       {sum(env_rewards):+.4f}')

In [ ]:
# Grid sweep over the 4 reward knobs. Pure CPU. Iterate freely.
import itertools
import pandas as pd
import numpy as np

GRID = {
    'reward_weight_performance': [0.4, 0.55, 0.7, 0.85, 1.0],
    'reward_weight_inflation':   [0.0, 0.15, 0.3, 0.5],
    'annual_inflation_rate':     [0.02, 0.05, 0.08],
    'transaction_cost_rate':     [0.0, 0.001, 0.005],
}

rows = []
for combo in itertools.product(*GRID.values()):
    cfg = dict(zip(GRID.keys(), combo))
    s = Settings(**cfg)
    rep = replay(trace, s)
    pos = sum(1 for r in rep['rewards_per_step'] if r > 0)
    rows.append({
        **cfg,
        'total': rep['total_reward'],
        'mean':  np.mean(rep['rewards_per_step']),
        'std':   np.std(rep['rewards_per_step']),
        'pos_steps': pos,
        'traj_bonus': rep['trajectory_bonus'],
    })

df = pd.DataFrame(rows).sort_values('total', ascending=False).reset_index(drop=True)
df.to_csv(RUN_DIR / 'sweep_results.csv', index=False)
print(f'{len(df)} combos. Top 8 by total reward:')
print(df.head(8).to_string(index=False))
print('\nBottom 5:')
print(df.tail(5).to_string(index=False))

In [ ]:
# Plots for the default-settings run (the one already saved in trace).
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter

steps = trace['steps']
x = list(range(len(steps)))
env_rewards = [s['env_reward'] for s in steps]
cum = np.cumsum(env_rewards)
portfolio = [s['portfolio_after'] for s in steps]
ideal_pct = trace['ideal_pnl_pct_series'][:len(steps)]
real_pct = [s['reward_breakdown'].get('real_pnl_pct', 0) for s in steps]
nominal_pct = [s['reward_breakdown'].get('nominal_pnl_pct', 0) for s in steps]

# 1) Cumulative reward
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, cum, label='cumulative env reward')
ax.axhline(0, color='gray', lw=0.5)
ax.set_title(f'{TASK}: cumulative reward vs step')
ax.set_xlabel('step'); ax.set_ylabel('cum reward'); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(RUN_DIR / 'reward_curve.png', dpi=120); plt.close(fig)

# 2) Ideal vs Real vs Nominal PnL
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, ideal_pct, label='ideal pnl %', linestyle='--')
ax.plot(x, nominal_pct, label='nominal pnl %')
ax.plot(x, real_pct, label='real (post-inflation) pnl %')
ax.axhline(0, color='gray', lw=0.5)
ax.set_title(f'{TASK}: pnl trajectory'); ax.legend(); ax.grid(alpha=0.3)
ax.set_xlabel('step'); ax.set_ylabel('pnl fraction')
fig.tight_layout(); fig.savefig(RUN_DIR / 'pnl_breakdown.png', dpi=120); plt.close(fig)

# 3) Action distribution
actions = [s['action']['side'] for s in steps]
qty = [s['action']['quantity'] for s in steps]
side_counts = Counter(actions)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(side_counts.keys(), side_counts.values())
axes[0].set_title('side distribution')
axes[1].hist(qty, bins=20)
axes[1].set_title('quantity distribution')
fig.tight_layout(); fig.savefig(RUN_DIR / 'actions.png', dpi=120); plt.close(fig)

from IPython.display import Image, display
for p in (RUN_DIR / 'reward_curve.png', RUN_DIR / 'pnl_breakdown.png', RUN_DIR / 'actions.png'):
    print(p); display(Image(str(p)))

In [ ]:
# Optional: exercise the extra-data path with one corpus task.
import sys
from app.data import corpus as _corpus
from app.core.corpus_tasks import list_corpus_task_ids
from tqdm import tqdm

RUN_CORPUS_STAGE = True   # flip False to skip even if corpus is built

if not RUN_CORPUS_STAGE:
    print('corpus stage: skipped (RUN_CORPUS_STAGE=False)')
elif not _corpus.available():
    print('corpus stage: skipped — data/corpus/ missing.')
    print('  build with:  python scripts/build_corpus.py --quick')
else:
    cids = list_corpus_task_ids()
    pick = next((c for c in cids if 'aapl' in c), cids[0])
    print(f'corpus stage: running {pick} ...')

    from app.core.environment import StockerEnv
    env = StockerEnv(pick)
    obs = env.reset().observation
    total_steps = env.state().total_steps
    
    n = 0; t0 = time.time()
    pbar = tqdm(total=total_steps, desc=pick, unit='step', file=sys.stdout, dynamic_ncols=True)
    while True:
        d = council.run(obs)
        r = env.step(d.action)
        n += 1
        pbar.update(1)
        pbar.set_postfix_str(
            f'{d.action.side}({d.action.quantity:+.0f}) {obs.date} '
            f'port={r.info.get("portfolio_value", 0):.0f}'
        )
        if r.done: break
        obs = r.observation
    pbar.close()
    
    print(f'\n  {pick}: {n} steps in {(time.time()-t0)/60:.1f} min, '
          f'final portfolio={r.info.get("portfolio_value"):.2f}')
    print('  loader fallback paths exercised: lookup_indicators / lookup_headlines / chart_path')

In [ ]:
# Pick a row from the sweep DataFrame and freeze it as the chosen settings.
# Default = top-1 by total reward. Edit `CHOSEN_INDEX` to pick another row
# (e.g. df.iloc[3] to override).
import json

CHOSEN_INDEX = 0   # 0 = top of the sorted sweep df; bump to any int to override

row = df.iloc[CHOSEN_INDEX]
chosen = {
    'reward_weight_performance': float(row['reward_weight_performance']),
    'reward_weight_inflation':   float(row['reward_weight_inflation']),
    'annual_inflation_rate':     float(row['annual_inflation_rate']),
    'transaction_cost_rate':     float(row['transaction_cost_rate']),
    '_total_reward_on_easy':     float(row['total']),
    '_chosen_at':                datetime.utcnow().isoformat(timespec='seconds') + 'Z',
}
out = RUN_DIR / 'chosen_settings.json'
out.write_text(json.dumps(chosen, indent=2))
print(f'wrote {out}')
print(json.dumps(chosen, indent=2))
print('\nTo apply during the next GRPO run, set env vars:')
for k, v in chosen.items():
    if k.startswith('_'): continue
    print(f'  export STOCKER_{k.upper()}={v}')

## Done

If every cell above ran without exception:
- The replay matches the env reward to <1e-5 → reward formula is captured exactly.
- The sweep gave you concrete weight tuples sorted by total reward.
- The corpus stage (if enabled) confirmed the loader fallbacks work end-to-end with real Gemma.
- `training/runs/easy_gemma4_<ts>/chosen_settings.json` holds the knobs for the next GRPO run.

**Next step**: re-run the corresponding GRPO cell in `train_grpo.ipynb` after
exporting the `STOCKER_*` env vars printed above (or paste them into
`app/config.py` defaults if you want them permanent).